# ツールの応用事例

## 1、args_schema を使用

例

In [1]:
from dotenv import load_dotenv

# .envファイルから環境変数を読み込む
load_dotenv(override=True)
from langchain.chat_models import init_chat_model
import os
from rich import print

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL")

model = init_chat_model(
    model=DEEPSEEK_MODEL,
    api_key=DEEPSEEK_API_KEY,
)


In [2]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool


class WeacherSchema(BaseModel):
    city: str = Field(default="北京", description="具体的な都市名")
    if_forecast: bool = Field(default=False, description="翌日の天気を含めるかどうか")


# ツールを定義
@tool("get_weacher_and_forecast", description="当日の天気を検索し、明日の天気予報を含めることができる",
      args_schema=WeacherSchema)
def get_weather(city: str, if_forecast: bool):
    res = f"{city}は今日良い天気です"
    if if_forecast:
        res += "\n明日は雨です"
    return res


print(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weacher_and_forecast',
        'description': '当日の天気を検索し、明日の天気予報を含めることができる',
        'parameters': {
            'properties': {
                'city': {'default': '北京', 'description': '具体的な都市名', 'type': 'string'},
                'if_forecast': {'default': False, 'description': '翌日の天気を含めるかどうか', 'type': 'boolean'}
            },
            'type': 'object'
        }
    }
}

In [3]:
from langchain_core.messages import HumanMessage

# 1、ツールをモデルにバインド
model_with_tools = model.bind_tools([get_weather])

# 2、メッセージリストを維持
messages = [HumanMessage("今日の杭州の天気はどうですか？明日は？")]

# 3、モデルを呼び出し、レスポンス：AIMessage を取得
response = model_with_tools.invoke(messages)

messages.append(response)

# 4、レスポンス内の tool_calls フィールド情報を取得
tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weacher_and_forecast":
        # 5、ツールを呼び出す（大規模言語モデルは直接ツールを呼び出せないため、ここでは明示的にツール呼び出しを実行する）
        # 呼び出し完了後、ToolMessage のインスタンスが返される
        tool_message = get_weather.invoke(tool_call)
        messages.append(tool_message)

# 6、モデルを呼び出し、AIMessage を取得
final_response = model.invoke(messages)

# 7、メッセージリストに追加
messages.append(final_response)

# 8、メッセージリストを走査
for msg in messages:
    msg.pretty_print()


================================ Human Message =================================

今日の杭州の天気はどうですか？明日は？
================================== Ai Message ==================================

お問い合わせいただきありがとうございます。杭州の天気を調べてみますね。
Tool Calls:
  get_weacher_and_forecast (call_00_BjBTqYrGX9c67nN04K4U7760)
 Call ID: call_00_BjBTqYrGX9c67nN04K4U7760
  Args:
    city: 杭州
    if_forecast: True
================================= Tool Message =================================
Name: get_weacher_and_forecast

杭州は今日良い天気です
明日は雨です
================================== Ai Message ==================================

杭州の天気は以下の通りです。

- **今日**：良い天気です。
- **明日**：雨が降る予報です。

何か他に気になることがあれば、お気軽にお尋ねください。


## 2、docstring を書く

In [4]:

from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool


# ツールを定義
@tool("get_weacher_and_forecast", parse_docstring=True)
def get_weather(city: str = "北京", if_forecast: bool = False):
    """
    当日の天気を検索し、明日の天気予報を含めることができる

    Args:
        city : 都市名
        if_forecast : 明日の天気を含めるかどうか
    """
    res = f"{city}は今日良い天気です"
    if if_forecast:
        res += "\n明日は雨です"
    return res


print(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weacher_and_forecast',
        'description': '当日の天気を検索し、明日の天気予報を含めることができる',
        'parameters': {
            'properties': {
                'city': {'default': '北京', 'description': '都市名', 'type': 'string'},
                'if_forecast': {'default': False, 'description': '明日の天気を含めるかどうか', 'type': 'boolean'}
            },
            'type': 'object'
        }
    }
}

In [5]:
from langchain_core.messages import HumanMessage

# 1、ツールをモデルにバインド
model_with_tools = model.bind_tools([get_weather])

# 2、メッセージリストを維持
messages = [HumanMessage("今日の杭州の天気はどうですか？明日は？")]

# 3、モデルを呼び出し、レスポンス：AIMessage を取得
response = model_with_tools.invoke(messages)

messages.append(response)

# 4、レスポンス内の tool_calls フィールド情報を取得
tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weacher_and_forecast":
        # 5、ツールを呼び出す（大規模言語モデルは直接ツールを呼び出せないため、ここでは明示的にツール呼び出しを実行する）
        # 呼び出し完了後、ToolMessage のインスタンスが返される
        tool_message = get_weather.invoke(tool_call)
        messages.append(tool_message)

# 6、モデルを呼び出し、AIMessage を取得
final_response = model.invoke(messages)

# 7、メッセージリストに追加
messages.append(final_response)

# 8、メッセージリストを走査
for msg in messages:
    msg.pretty_print()


================================ Human Message =================================

今日の杭州の天気はどうですか？明日は？
================================== Ai Message ==================================
Tool Calls:
  get_weacher_and_forecast (call_00_XoccLTuczR5eFu85y6wa8465)
 Call ID: call_00_XoccLTuczR5eFu85y6wa8465
  Args:
    city: 杭州
    if_forecast: True
================================= Tool Message =================================
Name: get_weacher_and_forecast

杭州は今日良い天気です
明日は雨です
================================== Ai Message ==================================

杭州の今日の天気は良い天気で、明日は雨の予報です。


## 3、複数ツールの呼び出し

In [6]:

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from rich import print as rprint


# 1. ツールを定義
# 株価検索ツールを定義
@tool(parse_docstring=True)
def get_stock_price(company: str, timeframe: str = "today") -> str:
    """指定した会社の株価情報を取得する

    Args:
        company: 会社名（例：Apple社、Microsoft社、Google社）
        timeframe: 期間（today-本日、week-今週、month-今月）
    """
    # 株価データのモック
    mock_data = {
        "Apple社": {"today": 185.20, "week": 183.50, "month": 180.75},
        "Microsoft社": {"today": 415.86, "week": 412.30, "month": 405.42},
        "Google社": {"today": 15.42, "week": 15.20, "month": 14.85}
    }

    if company in mock_data:
        price = mock_data[company].get(timeframe, "不明な期間")
        return f"{company} {timeframe}価格: {price}ドル"
    else:
        return f"ティッカー {company} のデータが見つかりません"


# ニュース検索ツールを定義
@tool(parse_docstring=True)
def search_news(company: str) -> str:
    """指定した会社の経済ニュースを検索する

    Args:
        company: 会社名

    Returns:
        会社の経済ニュース。1行につき1件
    """
    # ニュースデータのモック
    mock_news = {
        "Apple社": [
            "Appleが新型iPhoneを発表、株価が3%上昇",
            "Appleが欧州連合と独占禁止法の和解合意に達する",
            "Appleがインドでの生産規模を拡大へ"
        ],
        "Microsoft社": [
            "MicrosoftのAzureクラウド事業、四半期成長が予想を上回る",
            "MicrosoftがNuanceの買収を完了",
            "Microsoftが次世代AIアシスタントCopilotを発表"
        ],
        "Google社": [
            "Googleが新しいAIモデルを発表、性能が20%向上",
            "GoogleがOpenAIと提携し、新しいAIアシスタントを開発",
            "GoogleがヨーロッパでAI研究プロジェクトを展開"
        ]
    }

    news_list = mock_news.get(company, [f"{company}に関するニュースが見つかりません"])
    return "\n".join(news_list)


# rprint(convert_to_openai_tool(search_news))

# 2. モデルを初期化してツールをバインド
tools = [get_stock_price, search_news]
model_with_tools = model.bind_tools(tools)

message_list = []
human_message = HumanMessage(content="Apple社の本日の株価はいくらですか？最近のニュースは何がありますか？")
# human_message = HumanMessage(content="MicrosoftとAppleの株価を比較してください")
# human_message = HumanMessage(content="Tencentの最近の重要なニュースは何ですか？")
# human_message = HumanMessage(content="海水はなぜ塩辛いのですか？")
message_list.append(human_message)

# 3. ツール呼び出し
while True:
    response = model_with_tools.invoke(message_list)

    rprint(response)
    # break
    # 返された AIMessage をメッセージリストに追加
    message_list.append(response)

    # モデルがツールを呼び出す必要がない場合、そのままループを終了
    if not response.tool_calls:
        print("ツール呼び出しなし、そのまま回答を返す")
        break
# 
    # ツール呼び出しがある場合、ツール呼び出しのレスポンスを処理
    # 4. 開発者はモデルのレスポンスに基づき、ツールを呼び出して結果を取得
    for tool_call in response.tool_calls:
        if tool_call["name"] == "get_stock_price":
            stock_result = get_stock_price.invoke(tool_call)
            print("stock_result", stock_result)
            message_list.append(stock_result)
        if tool_call["name"] == "search_news":
            news_result = search_news.invoke(tool_call)
            print("news_result", news_result)
            message_list.append(news_result)

# 
# # print("response", response)
# # print(response.content)
# 
for msg in message_list:
    msg.pretty_print()

AIMessage(
    content='了解しました！Apple社の本日の株価と最近のニュースを同時に取得します。',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': "The user is asking for Apple's stock price today and recent news. I'll make both 
calls at the same time since they're independent."
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 144,
            'prompt_tokens': 416,
            'total_tokens': 560,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 28,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 416
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-pro',
        'system_fingerprint': 'fp_9954b31ca7_prod0820_fp8_kvcache_20260402',
        'id': 'd43e9559-1d06-4488-a334-d7a664267fb8',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--019fc171-9824-72b0-986f-a63cd7c4db6c-0',
    tool_calls=[
        {
            'name': 'get_stock_price',
            'args': {'company': 'Apple社', 'timeframe': 'today'},
            'id': 'call_00_j0egF1cX295xFkigNKGg1166',
            'type': 'tool_call'
        },
        {
            'name': 'search_news',
            'args': {'company': 'Apple社'},
            'id': 'call_01_7R7L1kFQcbTUc4cRaGgd5155',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 416,
        'output_tokens': 144,
        'total_tokens': 560,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 28}
    }
)

stock_result
ToolMessage(
    content='Apple社 today価格: 185.2ドル',
    name='get_stock_price',
    tool_call_id='call_00_j0egF1cX295xFkigNKGg1166'
)

news_result
ToolMessage(
    content='Appleが新型iPhoneを発表、株価が3%上昇\nAppleが欧州連合と独占禁止法の和解合意に達する\nAppleがインドで
の生産規模を拡大へ',
    name='search_news',
    tool_call_id='call_01_7R7L1kFQcbTUc4cRaGgd5155'
)

AIMessage(
    content='以下がApple社に関する最新情報です！\n\n---\n\n## 📈 株価情報（本日）\n\n| 項目 | 内容 
|\n|------|------|\n| **会社名** | Apple社 |\n| **本日の株価** | **185.2ドル** |\n\n---\n\n## 📰 
最近のニュース\n\n1. **🍎 Appleが新型iPhoneを発表、株価が3%上昇**\n   
新型iPhoneの発表が市場で好意的に受け止められ、株価の上昇につながったようです。\n\n2. **⚖️ 
Appleが欧州連合と独占禁止法の和解合意に達する**\n   
EUとの間で続いていた独占禁止法に関する問題について和解が成立しました。\n\n3. **🏭 
Appleがインドでの生産規模を拡大へ**\n   
サプライチェーン多様化の一環として、インドでの生産能力をさらに拡大する方針です。\n\n---\n\n全体的にポジティブなニュ
ースが多く、特に新型iPhoneの発表が株価に良い影響を与えているようですね。さらに詳しい情報が必要であればお知らせくだ
さい！',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'I have both results. Let me summarize them for the user in Japanese.'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 262,
            'prompt_tokens': 636,
            'total_tokens': 898,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 15,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 512},
            'prompt_cache_hit_tokens': 512,
            'prompt_cache_miss_tokens': 124
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-pro',
        'system_fingerprint': 'fp_9954b31ca7_prod0820_fp8_kvcache_20260402',
        'id': '4eb4d0f9-3dce-49be-9824-87f8163599e2',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019fc171-a257-74c2-b373-2995d88bc70f-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 636,
        'output_tokens': 262,
        'total_tokens': 898,
        'input_token_details': {'cache_read': 512},
        'output_token_details': {'reasoning': 15}
    }
)

ツール呼び出しなし、そのまま回答を返す

================================ Human Message =================================

Apple社の本日の株価はいくらですか？最近のニュースは何がありますか？
================================== Ai Message ==================================

了解しました！Apple社の本日の株価と最近のニュースを同時に取得します。
Tool Calls:
  get_stock_price (call_00_j0egF1cX295xFkigNKGg1166)
 Call ID: call_00_j0egF1cX295xFkigNKGg1166
  Args:
    company: Apple社
    timeframe: today
  search_news (call_01_7R7L1kFQcbTUc4cRaGgd5155)
 Call ID: call_01_7R7L1kFQcbTUc4cRaGgd5155
  Args:
    company: Apple社
================================= Tool Message =================================
Name: get_stock_price

Apple社 today価格: 185.2ドル
================================= Tool Message =================================
Name: search_news

Appleが新型iPhoneを発表、株価が3%上昇
Appleが欧州連合と独占禁止法の和解合意に達する
Appleがインドでの生産規模を拡大へ
================================== Ai Message ==================================

以下がApple社に関する最新情報です！

---

## 📈 株価情報（本日）

| 項目 | 内容 |
|------|------|
| **会社名** | Apple社 |
| **本日の株価**

## 4、複数ツールの呼び出し

In [7]:
from langchain.tools import tool
from langchain.messages import HumanMessage


@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    当日の天気を取得する

    Args:
        city: 都市名
    """
    return f'{city}は本日晴れです'


@tool(parse_docstring=True)
def get_news() -> str:
    """
    当日のニュースを取得する
    """
    return "近ごろ、世界的な半導体不足など複数の要因の影響を受け、各地の買取業者によると中古スマートフォンの回収市場が“活況”を迎えており、買取価格が軒並み上昇し、中古スマートフォンが“引っ張りだこ”となっている。"


model_with_tools = model.bind_tools([get_weather, get_news])

messages = [
    HumanMessage("今日の杭州の天気はどうですか？今日のニュースは何ですか？でたらめを言わないでください")
]

response = model_with_tools.invoke(messages)
response.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_wKyAVBeOH6MbNbsOjFDy0173)
 Call ID: call_00_wKyAVBeOH6MbNbsOjFDy0173
  Args:
    city: 杭州
  get_news (call_01_1s4cGJJSlEhGSEXaSfI37872)
 Call ID: call_01_1s4cGJJSlEhGSEXaSfI37872
  Args:


In [8]:

messages.append(response)

for tool_call in response.tool_calls:
    if tool_call["name"] == "get_weather":
        tool_msg = get_weather.invoke(tool_call)
        print(tool_msg)
        messages.append(tool_msg)
    elif tool_call["name"] == "get_news":
        tool_msg = get_news.invoke(tool_call)
        print(tool_msg)
        messages.append(tool_msg)
    else:
        raise Exception("存在しないツールです")

final_response = model.invoke(messages)
messages.append(final_response)

for msg in messages:
    msg.pretty_print()

ToolMessage(content='杭州は本日晴れです', name='get_weather', tool_call_id='call_00_wKyAVBeOH6MbNbsOjFDy0173')

ToolMessage(
    content='近ごろ、世界的な半導体不足など複数の要因の影響を受け、各地の買取業者によると中古スマートフォンの回収市
場が“活況”を迎えており、買取価格が軒並み上昇し、中古スマートフォンが“引っ張りだこ”となっている。',
    name='get_news',
    tool_call_id='call_01_1s4cGJJSlEhGSEXaSfI37872'
)

================================ Human Message =================================

今日の杭州の天気はどうですか？今日のニュースは何ですか？でたらめを言わないでください
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_wKyAVBeOH6MbNbsOjFDy0173)
 Call ID: call_00_wKyAVBeOH6MbNbsOjFDy0173
  Args:
    city: 杭州
  get_news (call_01_1s4cGJJSlEhGSEXaSfI37872)
 Call ID: call_01_1s4cGJJSlEhGSEXaSfI37872
  Args:
================================= Tool Message =================================
Name: get_weather

杭州は本日晴れです
================================= Tool Message =================================
Name: get_news

近ごろ、世界的な半導体不足など複数の要因の影響を受け、各地の買取業者によると中古スマートフォンの回収市場が“活況”を迎えており、買取価格が軒並み上昇し、中古スマートフォンが“引っ張りだこ”となっている。
================================== Ai Message ==================================

今日の杭州の天気は晴れです。

今日のニュースです。世界的な半導体不足などの影響で、各地で中古スマートフォンの買取市場が活況を呈しており、買取価格が上昇して「引っ張りだこ」の状態になっているそうです。
